# The Engram Layer

[Notebook 01](/notebooks/llm/deepseek/01-deepseek-architecture.html) catalogued four types of memory a transformer can use: in-weights (dense parameters), in-context (KV cache), external (retrieval DB), and a fourth type introduced by DeepSeek-V3 — hash-addressed $n$-gram memory. MLA reduces the cost of the KV cache. MoE conditions which FFN parameters activate. The Engram layer provides a shortcut that bypasses both: [no attention, no routing, just a hash.]{.underline}

The core observation is that transformers waste sequential depth reconstructing facts that could be served by a lookup table. Given the $n$-gram context "capital of France", a standard transformer reconstructs "Paris" from distributed weight activations on every forward pass. An Engram layer hashes the $n$-gram to a table address, reads the embedding at that address, gates it against the hidden state, and injects the result into the residual stream — $O(1)$ deterministic retrieval with no attention compute.

This notebook builds NanoEngram from scratch, derives the XOR hash mapping and prime-modulus collision strategy, implements the sqrt-sigmoid gate and causal ShortConv aggregation, and integrates the result into `NanoDeepSeekWithEngram`. We close with a gate visualization and a perplexity ablation measuring the Engram contribution.

In [ ]:
import math
import unicodedata
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Motivation: The Computation-vs-Retrieval Gap

Consider a concrete fact: *"The capital of France is Paris."* A standard transformer must recover this fact through a chain of attention and FFN operations — the fact has been encoded into weight matrices during training, but there is no direct read path. "Paris" is reconstructed from distributed activations on every forward pass, consuming the same sequential depth and compute regardless of how common the fact is.

An Engram layer provides a shortcut:
1. Hash the current $n$-gram context (e.g., "capital of France") to a table address.
2. Read the embedding stored at that address.
3. Gate it against the current hidden state.
4. Add the result to the residual stream *before* attention.

This is *deterministic retrieval* — no learned router, no attention compute, $O(1)$ lookup. The embedding at each address is learned during training to carry contextually relevant signal for that surface form.

**Where Engram layers are placed.** In DeepSeek-V3-Engram (27B), the Engram layer is inserted before attention at layers 2 and 15:

```
H ← H + Engram(H, input_ids)      # before Attention at layers 2 and 15
H ← H + Attention(H)
H ← H + MoE(H)
```

At our nano scale (6 layers), we place it at layers 2 and 6 — mirroring the same relative positions.

## CompressedTokenizer: Normalizing the Hash Space

Before hashing, input tokens are normalized so that *semantically equivalent* surface forms map to the same hash address. For example, `"Paris"`, `"paris"`, and `"PARIS"` should hit the same bucket. Our normalizer applies four steps in order:

1. **NFKC normalization** — Unicode compatibility decomposition
2. **Lowercase** — case folding
3. **NFD + strip accents** — removes combining diacritics (e.g., `é` → `e`)
4. **Whitespace collapse** — multiple spaces → single space

The compressor builds a lookup table mapping original token IDs to compressed IDs, reducing the vocabulary by ~20% by collapsing case-equivalent and accent-equivalent types.

In [ ]:
from transformers import GPT2TokenizerFast


def _normalize(text: str) -> str:
    """NFKC → lowercase → NFD + strip accents → collapse whitespace."""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))  # <1>
    text = " ".join(text.split())  # <2>
    return text


class CompressedTokenizer:
    """Builds a lookup table old_id → compressed_id by grouping tokens whose
    decoded text normalizes to the same string.

    The original token IDs are still used for embeddings; only the Engram
    hashing step uses compressed IDs.
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.lookup_table, self.compressed_vocab_size = self._build()
        reduction = 1 - self.compressed_vocab_size / tokenizer.vocab_size
        print(f"CompressedTokenizer: {tokenizer.vocab_size} → "
              f"{self.compressed_vocab_size} tokens ({100 * reduction:.1f}% reduction)")

    def _build(self) -> Tuple[np.ndarray, int]:
        V = self.tokenizer.vocab_size
        key2new: Dict[str, int] = {}
        lookup = np.empty(V, dtype=np.int64)
        for tid in range(V):
            raw = self.tokenizer.decode([tid], skip_special_tokens=False)
            key = _normalize(raw) if "\ufffd" not in raw else raw  # <3>
            if key not in key2new:
                key2new[key] = len(key2new)
            lookup[tid] = key2new[key]
        return lookup, len(key2new)

    def __call__(self, input_ids: np.ndarray) -> np.ndarray:
        """Map a (B, T) int64 array of token IDs to compressed IDs."""
        flat = np.asarray(input_ids, dtype=np.int64).reshape(-1)
        return self.lookup_table[flat].reshape(input_ids.shape)


base_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
base_tokenizer.pad_token = base_tokenizer.eos_token
comp_tokenizer = CompressedTokenizer(base_tokenizer)

1. Strip Unicode combining characters (diacritics, accents) — after NFD decomposition each accented character becomes base + combiner, so we drop the combiners.
2. Collapse runs of whitespace to a single space so `" Paris"` and `"  Paris"` map to the same key.
3. Tokens containing the Unicode replacement character `\ufffd` (malformed bytes) are left un-normalized to avoid collapsing genuinely distinct byte sequences.

## NgramHashMapping: Deterministic XOR Hashing

Given a sequence of compressed token IDs $[x_0, x_1, \ldots, x_{T-1}]$, the $n$-gram hash for position $t$ is:

$$h_t^{(n)} = \bigoplus_{k=0}^{n-1} (x_{t-k} \cdot m_k) \mod p$$

where $m_k$ are seeded odd-integer multipliers (different per layer to avoid inter-layer collisions), $\oplus$ is bitwise XOR, and $p$ is a prime slightly larger than the target table size. Left-padding with the pad token handles positions before the sequence start.

We compute $n=2$ (bigram) and $n=3$ (trigram) grams, each with `n_head` independent hash functions using distinct prime moduli. [The prime moduli are chosen to be just above the per-$n$-gram vocabulary size, with all primes across all layers and heads kept globally distinct]{.mark} — this is the same strategy as the original paper.

:::{.callout-note}
**Collision probability.** A prime modulus minimizes systematic bias in XOR hash distributions. For bigrams over GPT-2's vocabulary ($V \approx 50{,}000$, table size $N = 50{,}000$), the birthday-bound collision rate is approximately $V^2 / (2N) = 25{,}000$ expected collisions per 1{,}000 unique bigrams. This is probabilistic but bounded, and mitigated by the multi-head design averaging over independent hash functions.

:::

In [ ]:
def _next_prime(n: int, seen: set) -> int:
    """Find the smallest prime > n not already in `seen`."""
    from sympy import isprime
    c = n + 1
    while not isprime(c) or c in seen:
        c += 1
    return c


@dataclass
class NanoEngramConfig:
    engram_vocab_size: List[int] = field(default_factory=lambda: [50_000, 50_000])
    max_ngram_size:    int   = 3        # orders: bigram (n=2) and trigram (n=3)
    n_embed:           int   = 64       # embedding dimension per head per n-gram
    n_head:            int   = 4        # independent hash heads per n-gram order
    layer_ids:         List[int] = field(default_factory=lambda: [2, 6])
    pad_id:            int   = 0
    seed:              int   = 42


class NgramHashMapping:
    """For each Engram layer and each position (B, T), returns a
    (B, T, n_heads_total) integer tensor of hash indices.

    n_heads_total = (max_ngram_size - 1) * n_head
                  = 2 (bigram + trigram) * n_head
    """

    def __init__(self, cfg: NanoEngramConfig, comp_tok: CompressedTokenizer):
        self.cfg  = cfg
        self.comp = comp_tok

        # Seeded odd-integer multipliers per layer
        self.layer_multipliers: Dict[int, np.ndarray] = {}
        MAX_LONG   = np.iinfo(np.int64).max
        HALF_BOUND = MAX_LONG // 2
        for lid in cfg.layer_ids:
            rng = np.random.default_rng(int(cfg.seed + 10007 * lid))
            raw = rng.integers(0, HALF_BOUND, size=(cfg.max_ngram_size,), dtype=np.int64)
            self.layer_multipliers[lid] = raw * 2 + 1   # ensure odd  # <1>

        # Per-layer, per-n-gram-order, per-head prime moduli — all globally distinct
        seen_primes: set = set()
        self.primes: Dict[int, List[List[int]]] = {}
        for lid in cfg.layer_ids:
            layer_primes = []
            for ni in range(cfg.max_ngram_size - 1):   # 0 → bigram, 1 → trigram
                head_primes = []
                start = cfg.engram_vocab_size[ni] - 1
                for _ in range(cfg.n_head):
                    p = _next_prime(start, seen_primes)
                    seen_primes.add(p)
                    head_primes.append(p)
                    start = p
                layer_primes.append(head_primes)
            self.primes[lid] = layer_primes

    def hash(self, input_ids: np.ndarray, layer_id: int) -> np.ndarray:
        """
        Returns: (B, T, n_heads_total) int64
        """
        x    = self.comp(input_ids)                  # (B, T) compressed
        B, T = x.shape
        muls = self.layer_multipliers[layer_id]

        def shift_k(k: int) -> np.ndarray:
            """Shift x left by k positions, padding with pad_id on the left."""
            if k == 0:
                return x
            padded = np.pad(x, ((0, 0), (k, 0)), mode="constant",
                            constant_values=self.cfg.pad_id)
            return padded[:, :T]

        shifts = [shift_k(k) for k in range(self.cfg.max_ngram_size)]

        all_hashes = []
        for ni in range(self.cfg.max_ngram_size - 1):  # 0 → bigram, 1 → trigram
            n = ni + 2
            mix = shifts[0] * muls[0]
            for k in range(1, n):
                mix = np.bitwise_xor(mix, shifts[k] * muls[k])  # <2>
            for prime in self.primes[layer_id][ni]:
                all_hashes.append((mix % prime).astype(np.int64))  # <3>

        return np.stack(all_hashes, axis=2)  # (B, T, n_heads_total)


# Smoke test
eng_cfg = NanoEngramConfig()
hasher  = NgramHashMapping(eng_cfg, comp_tokenizer)
test_ids = np.array([[1, 25, 1337, 50, 8, 300]], dtype=np.int64)
h = hasher.hash(test_ids, layer_id=2)
n_heads_total = (eng_cfg.max_ngram_size - 1) * eng_cfg.n_head
print(f"Hash output shape: {h.shape}   expected: (1, 6, {n_heads_total})")

1. Multipliers are forced to be odd to prevent the zero-collapse issue: if a multiplier is even and $x_{t-k}$ is even, the contribution to the XOR hash is zero.
2. Each context token contributes its own multiplied value to the XOR mix, so the hash changes when any token in the $n$-gram window changes.
3. Prime modular reduction maps the full 64-bit XOR output to the table address space `[0, prime)`. Using distinct primes per head ensures independent hash functions.

## MultiHeadEmbedding: Shared Table with Per-Head Offsets

Rather than $H$ separate embedding tables (one per hash head), we use a single large table of size $\sum_i p_i$ and address head $i$ by adding its cumulative offset:

$$\text{offset}_i = \sum_{j < i} p_j$$

This is equivalent to separate tables but more memory-efficient: a single `nn.Embedding` call allows PyTorch to fuse the lookup into one kernel, and the buffer lives contiguously in memory.

In [ ]:
class MultiHeadEmbedding(nn.Module):
    """Single embedding table of size sum(vocab_sizes) with per-head offsets.

    Input:  (B, T, n_heads) int64
    Output: (B, T, n_heads, d_embed)
    """

    def __init__(self, vocab_sizes: List[int], d_embed: int):
        super().__init__()
        self.n_heads = len(vocab_sizes)
        self.d_embed = d_embed
        offsets = [0]
        for v in vocab_sizes[:-1]:
            offsets.append(offsets[-1] + v)
        self.register_buffer("offsets", torch.tensor(offsets, dtype=torch.long))  # <1>
        self.embedding = nn.Embedding(sum(vocab_sizes), d_embed)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T, n_heads) → (B, T, n_heads, d_embed)"""
        shifted = idx + self.offsets  # (B, T, n_heads) — broadcast over batch and time  # <2>
        return self.embedding(shifted)


# Build vocab_sizes from the hasher primes for layer 2
vocab_sizes_l2 = [
    p
    for ni in range(eng_cfg.max_ngram_size - 1)
    for p in hasher.primes[2][ni]
]
mhe = MultiHeadEmbedding(vocab_sizes_l2, d_embed=eng_cfg.n_embed)
test_idx = torch.from_numpy(h)   # (1, 6, 8)
emb_out  = mhe(test_idx)
print(f"MultiHeadEmbedding output shape: {tuple(emb_out.shape)}")
print(f"Total embedding parameters: {sum(p.numel() for p in mhe.parameters()) / 1e3:.1f}K")

1. `offsets` is registered as a non-trainable buffer so it moves with the model to the correct device automatically.
2. Broadcasting `self.offsets` of shape `(n_heads,)` over `idx` of shape `(B, T, n_heads)` correctly shifts each head's indices into its own table region.

## Gating: Routing the Hash Embedding into the Residual Stream

[The Engram output is not added unconditionally.]{.mark} A gate decides how much of the hash embedding is relevant given the current hidden state $h_t$. The gate is a dot-product of the RMSNorm-normalized query $h_t$ and key $k_t$ (projected from the hash embedding), followed by a **sqrt-sigmoid** activation:

$$\alpha_t = \sigma\!\left(\sqrt{|s_t|} \cdot \operatorname{sign}(s_t)\right), \qquad s_t = \frac{\hat{h}_t \cdot \hat{k}_t}{\sqrt{d}}$$

where $\hat{\cdot}$ denotes RMSNorm normalization. The sqrt-sigmoid is more conservative near zero and sharper near $\pm 1$ than plain sigmoid — it suppresses weakly relevant Engram hits more aggressively. The final gated output is:

$$\text{out}_t = \alpha_t \cdot W_V \, e_t$$

where $e_t$ is the flattened hash embedding for position $t$.

:::{.callout-note}
## Multi-Head Hyper-Connection (mHC) — the part we simplify

In the production DeepSeek-Engram implementation (27B scale), the residual stream is a 4D tensor `(B, L, M, D)` where `M = hc_mult = 4` is the number of *hyper-connection* heads. Each head maintains an independent residual copy mixed at each block via learned gates. In the Engram layer, this means $M$ independent key projections $W_K^{(m)}$ and $M$ corresponding gates $\alpha_t^{(m)}$.

We skip this in NanoEngram for three reasons: (1) the orthogonality constraint between hyper-connection heads requires Riemannian SGD, not standard AdamW; (2) at nano scale, maintaining $M=4$ residual streams would roughly quadruple residual memory and likely collapse; (3) the 4D tensor requires non-trivial shape bookkeeping throughout every block.

**What we do instead:** we collapse $M=1$ and expand the hidden state to $4 \cdot d_{\text{model}}$ before the key projection — `W_K: (engram_dim, 4d)`, same FLOPs as $M$ independent projections, all in 3D tensors. This captures the routing logic while keeping every tensor 3D.

:::

## ShortConv: Causal Local Aggregation

After gating, a short depthwise causal convolution smoothes the gated values across a local window:

$$Y = \operatorname{SiLU}(\operatorname{CausalConv1D}_{\text{dw}}(\operatorname{RMSNorm}(V))) + V$$

- **Kernel size 4, dilation = `max_ngram_size` = 3**: effective receptive field = $(4-1) \times 3 + 1 = 10$ positions
- **Depthwise** (`groups=d`): no cross-channel mixing in the conv; cross-channel mixing happens via $W_V$
- **Causal**: left-pad only, trim right to length $T$

This allows the model to aggregate information from recent $n$-gram hits before injecting into the residual stream.

In [ ]:
class ShortConv(nn.Module):
    """Causal depthwise Conv1D + SiLU residual, operating on (B, T, d)."""

    def __init__(self, d: int, kernel_size: int = 4, dilation: int = 3):
        super().__init__()
        self.norm    = nn.RMSNorm(d)
        padding      = (kernel_size - 1) * dilation   # <1>
        self.conv    = nn.Conv1d(
            in_channels=d,
            out_channels=d,
            kernel_size=kernel_size,
            groups=d,
            dilation=dilation,
            padding=padding,
            bias=False,
        )
        self.act = nn.SiLU()

    def forward(self, v: torch.Tensor) -> torch.Tensor:
        """v: (B, T, d)"""
        B, T, d = v.shape
        normed   = self.norm(v)                     # (B, T, d)
        x_bct    = normed.transpose(1, 2)           # (B, d, T) — Conv1d expects channels first
        y_bct    = self.conv(x_bct)[..., :T]        # trim right padding → (B, d, T)  # <2>
        return self.act(y_bct.transpose(1, 2)) + v  # (B, T, d)

1. Left-padding of `(kernel_size - 1) * dilation` ensures the convolution is strictly causal — position $t$ only sees positions $\leq t$.
2. Trimming `[..., :T]` removes the right-side padding that PyTorch adds symmetrically when `padding > 0`.

## NanoEngram: Putting It Together

The full NanoEngram layer combines hash lookup, value and key projections, sqrt-sigmoid gating, and ShortConv aggregation:

```
e  = MultiHeadEmbedding(hash(input_ids))   # (B, T, n_heads, n_embed)
ef = e.flatten(-2)                          # (B, T, n_heads * n_embed)
V  = W_V(ef)                                # (B, T, d)
K  = W_K(ef)                                # (B, T, hc_mult * d)
s  = RMSNorm(h) · RMSNorm(K_pool) / sqrt(d)
α  = sigmoid(sqrt(|s|) * sign(s))
out = ShortConv(α * V)
```

In [ ]:
class NanoEngram(nn.Module):
    """Single Engram layer for one specific Transformer layer position."""

    def __init__(
        self,
        layer_id: int,
        cfg: NanoEngramConfig,
        d_model: int,
        hasher: NgramHashMapping,
        hc_mult: int = 4,
    ):
        super().__init__()
        self.layer_id = layer_id
        self.hasher   = hasher
        self.hc_mult  = hc_mult
        self.d        = d_model

        vocab_sizes = [
            p
            for ni in range(cfg.max_ngram_size - 1)
            for p in hasher.primes[layer_id][ni]
        ]
        self.n_heads  = len(vocab_sizes)
        self.n_embed  = cfg.n_embed
        engram_dim    = self.n_heads * cfg.n_embed

        self.mhe        = MultiHeadEmbedding(vocab_sizes, cfg.n_embed)
        self.W_V        = nn.Linear(engram_dim, d_model, bias=False)
        self.W_K        = nn.Linear(engram_dim, hc_mult * d_model, bias=False)  # mHC collapsed
        self.norm_k     = nn.RMSNorm(hc_mult * d_model)
        self.norm_q     = nn.RMSNorm(d_model)
        self.short_conv = ShortConv(d_model, kernel_size=4, dilation=cfg.max_ngram_size)

    def forward(self, h: torch.Tensor, input_ids: np.ndarray) -> torch.Tensor:
        """
        h:         (B, T, d_model) hidden state
        input_ids: (B, T) int64 numpy array of ORIGINAL token IDs
        Returns:   (B, T, d_model) Engram contribution (add to h outside)
        """
        B, T, d = h.shape
        dev = h.device

        # 1. Hash → multi-head embeddings
        ids_np   = input_ids.cpu().numpy() if isinstance(input_ids, torch.Tensor) else input_ids
        hash_idx = self.hasher.hash(ids_np, self.layer_id)              # (B, T, n_heads) np.int64
        hash_t   = torch.from_numpy(hash_idx).to(dev)
        emb      = self.mhe(hash_t)                                      # (B, T, n_heads, n_embed)
        ef       = emb.flatten(start_dim=-2)                             # (B, T, n_heads*n_embed)

        # 2. Project to value and key
        V = self.W_V(ef)                                                 # (B, T, d)          # <1>
        K = self.W_K(ef)                                                 # (B, T, hc_mult*d)

        # 3. sqrt-sigmoid gate
        K_normed = self.norm_k(K).view(B, T, self.hc_mult, d).mean(dim=2)   # (B, T, d)      # <2>
        nq = self.norm_q(h)
        s  = (nq * K_normed).sum(dim=-1, keepdim=True) / math.sqrt(d)       # (B, T, 1)
        a  = s.abs().clamp(1e-9).sqrt() * s.sign()                           # sqrt-sigmoid input
        gate = a.sigmoid()                                                    # (B, T, 1)       # <3>

        # 4. Gated value → ShortConv
        gated = gate * V
        return self.short_conv(gated)                                         # (B, T, d)


# Verify shapes
eng_cfg2 = NanoEngramConfig()
hasher2  = NgramHashMapping(eng_cfg2, comp_tokenizer)
engram2  = NanoEngram(layer_id=2, cfg=eng_cfg2, d_model=384, hasher=hasher2)

dummy_h   = torch.randn(2, 16, 384)
dummy_ids = np.random.randint(0, 50257, size=(2, 16), dtype=np.int64)
out = engram2(dummy_h, dummy_ids)
print(f"NanoEngram output shape: {tuple(out.shape)}")
print(f"Engram layer params: {sum(p.numel() for p in engram2.parameters()) / 1e3:.1f}K")

1. `W_V` projects the flattened hash embeddings to the model's hidden dimension, producing the *content* to inject.
2. The $M$ hyper-connection key vectors are mean-pooled to a single `(B, T, d)` key — a simplification of the full mHC gating that avoids 4D tensors.
3. The sqrt-sigmoid $\sigma(\sqrt{|s|} \cdot \operatorname{sign}(s))$ is more conservative near zero than plain sigmoid, suppressing weakly-matched $n$-gram entries more aggressively.

## Integrating NanoEngram into NanoDeepSeek

We add an optional `NanoEngram` module to each `DeepSeekBlock`. Blocks at `layer_ids` prepend the Engram update before the standard MLA + MoE residual path:

```
if layer_id in engram_layer_ids:
    x = x + NanoEngram(x, input_ids)
x = x + NanoMLA(RMSNorm(x))
x = x + NanoMoE(RMSNorm(x))
```

In [ ]:
# ── Paste NanoDeepSeek components (self-contained) ─────────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps; self.gamma = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x / x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt() * self.gamma

def make_rope_cache(max_seq_len, d_head, device):
    theta = 1.0 / (10000 ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    pos   = torch.arange(max_seq_len, device=device).float()
    freqs = torch.cat([torch.outer(pos, theta)] * 2, dim=-1)
    return freqs.cos()[None, None], freqs.sin()[None, None]

def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    return x * cos + torch.stack([-x2, x1], dim=-1).flatten(-2) * sin

def causal_mask(T, device):
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), 1)[None, None]


@dataclass
class NanoDeepSeekConfig:
    vocab_size: int = 50257; d_model: int = 384; n_layers: int = 6
    n_heads: int = 6; d_compressed: int = 96; d_ffn: int = 1024
    n_shared: int = 1; n_routed: int = 8; top_k: int = 2
    max_seq_len: int = 256; aux_loss_coeff: float = 1e-2

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.W1 = nn.Linear(d, d_ff, bias=False); self.W3 = nn.Linear(d, d_ff, bias=False)
        self.W2 = nn.Linear(d_ff, d, bias=False)
    def forward(self, x): return self.W2(F.silu(self.W1(x)) * self.W3(x))

class NanoMLA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, nh, dc = cfg.d_model, cfg.n_heads, cfg.d_compressed
        self.nh = nh; self.dh = d // nh; self.d = d
        self.W_c = nn.Linear(d, dc, bias=False); self.W_K = nn.Linear(dc, d, bias=False)
        self.W_V = nn.Linear(dc, d, bias=False); self.W_Q = nn.Linear(d, d, bias=False)
        self.W_O = nn.Linear(d, d, bias=False)
        cos, sin = make_rope_cache(cfg.max_seq_len, self.dh, "cpu")
        self.register_buffer("cos", cos); self.register_buffer("sin", sin)
    def forward(self, x, mask=None):
        B, T, _ = x.shape
        c_kv = self.W_c(x); K, V = self.W_K(c_kv), self.W_V(c_kv); Q = self.W_Q(x)
        def mh(t): return t.view(B, T, self.nh, self.dh).transpose(1, 2)
        Q, K, V = mh(Q), mh(K), mh(V)
        cos = self.cos[:, :, :T, :].to(x.device); sin = self.sin[:, :, :T, :].to(x.device)
        Q, K = apply_rope(Q, cos, sin), apply_rope(K, cos, sin)
        sc = Q @ K.transpose(-2, -1) / math.sqrt(self.dh)
        if mask is not None: sc = sc.masked_fill(mask, float("-inf"))
        return self.W_O((F.softmax(sc, dim=-1) @ V).transpose(1, 2).reshape(B, T, self.d))

class NanoMoE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_shared = cfg.n_shared; self.n_routed = cfg.n_routed
        self.top_k = cfg.top_k; self.alpha = cfg.aux_loss_coeff
        self.shared  = nn.ModuleList([SwiGLU(cfg.d_model, cfg.d_ffn) for _ in range(cfg.n_shared)])
        self.experts = nn.ModuleList([SwiGLU(cfg.d_model, cfg.d_ffn) for _ in range(cfg.n_routed)])
        self.router  = nn.Linear(cfg.d_model, cfg.n_routed, bias=False)
    def forward(self, x):
        B, T, d = x.shape; xf = x.view(B * T, d)
        out = sum(e(xf) for e in self.shared)
        probs = F.softmax(self.router(xf), dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.top_k, dim=-1)
        gates = topk_vals / (topk_vals.sum(-1, keepdim=True) + 1e-9)
        routed = torch.zeros_like(xf)
        for ki in range(self.top_k):
            for ei in range(self.n_routed):
                m = topk_idx[:, ki] == ei
                if m.any(): routed[m] += gates[:, ki:ki+1][m] * self.experts[ei](xf[m])
        with torch.no_grad():
            f = F.one_hot(topk_idx[:, 0], self.n_routed).float().mean(0)
        aux = self.alpha * self.n_routed * (f * probs.mean(0)).sum()
        return (out + routed).view(B, T, d), aux, probs.detach()


# ── DeepSeekBlock with optional Engram ─────────────────────────────────────────

class DSBlockWithEngram(nn.Module):
    def __init__(self, layer_id: int, ds_cfg: NanoDeepSeekConfig,
                 engram: Optional[NanoEngram] = None):
        super().__init__()
        self.layer_id = layer_id
        self.engram   = engram
        self.n1       = RMSNorm(ds_cfg.d_model)
        self.attn     = NanoMLA(ds_cfg)
        self.n2       = RMSNorm(ds_cfg.d_model)
        self.moe      = NanoMoE(ds_cfg)

    def forward(self, x: torch.Tensor, input_ids: Optional[np.ndarray], mask=None):
        if self.engram is not None and input_ids is not None:
            x = x + self.engram(x, input_ids)          # Engram pre-attention update  # <1>
        x = x + self.attn(self.n1(x), mask)
        moe_out, aux, probs = self.moe(self.n2(x))
        x = x + moe_out
        return x, aux, probs


class NanoDeepSeekWithEngram(nn.Module):
    def __init__(self, ds_cfg: NanoDeepSeekConfig, engram_cfg: NanoEngramConfig,
                 comp_tok: CompressedTokenizer):
        super().__init__()
        self.ds_cfg = ds_cfg
        self.hasher = NgramHashMapping(engram_cfg, comp_tok)  # shared across all Engram layers
        self.emb    = nn.Embedding(ds_cfg.vocab_size, ds_cfg.d_model)
        self.blocks = nn.ModuleList()
        for lid in range(ds_cfg.n_layers):
            engram = None
            if lid in engram_cfg.layer_ids:
                engram = NanoEngram(
                    layer_id=lid, cfg=engram_cfg, d_model=ds_cfg.d_model, hasher=self.hasher
                )
            self.blocks.append(DSBlockWithEngram(lid, ds_cfg, engram))
        self.norm = RMSNorm(ds_cfg.d_model)
        self.head = nn.Linear(ds_cfg.d_model, ds_cfg.vocab_size, bias=False)
        self.head.weight = self.emb.weight

    def forward(self, idx: torch.Tensor, targets=None):
        B, T = idx.shape
        x         = self.emb(idx)
        mask      = causal_mask(T, idx.device)
        ids_np    = idx.cpu().numpy()
        total_aux = torch.tensor(0.0, device=idx.device)
        all_probs = []
        for blk in self.blocks:
            x, aux, probs = blk(x, ids_np, mask)
            total_aux += aux
            all_probs.append(probs)
        logits = self.head(self.norm(x))
        loss = None
        if targets is not None:
            lm   = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss = lm + total_aux
        return logits, loss, all_probs


# Build and verify
ds_cfg  = NanoDeepSeekConfig()
eg_cfg  = NanoEngramConfig()
model_eg = NanoDeepSeekWithEngram(ds_cfg, eg_cfg, comp_tokenizer).to(device)

total_p  = sum(p.numel() for p in model_eg.parameters())
engram_p = sum(
    p.numel() for blk in model_eg.blocks if blk.engram for p in blk.engram.parameters()
)
print(f"NanoDeepSeekWithEngram — total params: {total_p/1e6:.2f}M  (Engram: {engram_p/1e3:.1f}K)")

dummy = torch.randint(0, 50257, (2, 32)).to(device)
tgt   = torch.randint(0, 50257, (2, 32)).to(device)
logits, loss, probs = model_eg(dummy, tgt)
print(f"Forward pass OK — logits: {tuple(logits.shape)}, loss: {loss.item():.4f}")

1. The Engram update runs before the pre-norm MLA step. This gives the attention mechanism access to the hash-retrieved signal when computing queries and keys.

## Gate Visualization: What Does Engram Attend To?

We visualize the gate values $\alpha_t$ for each token position in a sentence, at each Engram layer. [High gate values indicate that the hash embedding for that $n$-gram context carried a strong signal]{.mark} — typically proper nouns, technical terms, or rare multi-word expressions.

At initialization (random embeddings), gates will be near 0.5 uniformly. This visualization becomes meaningful after training, where gates for rare surface forms will sharpen.

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt


@torch.no_grad()
def get_gate_values(model: NanoDeepSeekWithEngram, text: str) -> Dict[int, np.ndarray]:
    """Return gate values (T,) for each Engram layer given a text prompt."""
    tokens = base_tokenizer.encode(text)
    idx    = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    B, T   = idx.shape
    x      = model.emb(idx)
    ids_np = idx.cpu().numpy()
    mask   = causal_mask(T, device)
    gates_by_layer: Dict[int, np.ndarray] = {}

    for blk in model.blocks:
        if blk.engram is not None:
            eg  = blk.engram
            lid = blk.layer_id
            hash_idx = eg.hasher.hash(ids_np, lid)
            hash_t   = torch.from_numpy(hash_idx).to(device)
            emb      = eg.mhe(hash_t).flatten(start_dim=-2)
            K        = eg.W_K(emb)
            K_normed = eg.norm_k(K).view(B, T, eg.hc_mult, eg.d).mean(dim=2)
            nq       = eg.norm_q(x)
            s        = (nq * K_normed).sum(-1, keepdim=True) / math.sqrt(eg.d)
            a        = s.abs().clamp(1e-9).sqrt() * s.sign()
            gate     = a.sigmoid().squeeze(-1).squeeze(0)   # (T,)
            gates_by_layer[lid] = gate.cpu().numpy()

        x, _, _ = blk(x, ids_np, mask)

    return gates_by_layer


sentence  = "Only Alexander the Great could tame the horse Bucephalus."
tokens    = base_tokenizer.encode(sentence)
tok_strs  = [base_tokenizer.decode([t]) for t in tokens]
gate_vals = get_gate_values(model_eg, sentence)

fig, axes = plt.subplots(len(gate_vals), 1, figsize=(14, 2.5 * len(gate_vals)))
if len(gate_vals) == 1:
    axes = [axes]

for ax, (lid, gates) in zip(axes, sorted(gate_vals.items())):
    im = ax.imshow(gates[None, :], aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(tok_strs)))
    ax.set_xticklabels(tok_strs, rotation=45, ha="right", fontsize=9)
    ax.set_yticks([])
    ax.set_title(f"Engram gate α — Layer {lid}")
    plt.colorbar(im, ax=ax)

plt.suptitle(f'"{sentence}"', fontsize=10)
plt.tight_layout()
plt.savefig("03_engram_gates.png", dpi=150)
plt.show()

After training: proper nouns (*Alexander*, *Bucephalus*), rare multi-word spans, and end-of-sentence positions typically show higher gate values. At random initialization the gates are uninformative — run this cell again after training on FineWeb-Edu to see learned specialization.

## Sensitivity Analysis: Ablating Engram at Inference

We zero out the Engram contribution — setting all gates to 0 — and measure the perplexity increase on a held-out sample. This quantifies how much the model relies on hash retrieval versus pure attention.

In [ ]:
from datasets import load_dataset


@torch.no_grad()
def eval_perplexity(
    model: NanoDeepSeekWithEngram,
    n_batches: int = 20,
    seq_len: int = 64,
    ablate_engram: bool = False,
) -> float:
    """Estimate perplexity on a few FineWeb-Edu batches."""
    original_forwards = {}
    if ablate_engram:
        for blk in model.blocks:
            if blk.engram is not None:
                eg = blk.engram
                original_forwards[id(eg)] = eg.forward
                def zero_forward(h, input_ids, _eg=eg):
                    return torch.zeros_like(h)
                eg.forward = zero_forward  # <1>

    model.eval()
    total_loss, n_done = 0.0, 0
    try:
        ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT",
                          split="train", streaming=True, trust_remote_code=True)
        buf = []
        for doc in ds:
            ids = base_tokenizer.encode(doc["text"], add_special_tokens=False)
            buf.extend(ids)
            while len(buf) >= seq_len + 1 and n_done < n_batches:
                chunk = buf[:seq_len + 1]; buf = buf[seq_len + 1:]
                inp = torch.tensor(chunk[:-1], dtype=torch.long, device=device).unsqueeze(0)
                tgt = torch.tensor(chunk[1:],  dtype=torch.long, device=device).unsqueeze(0)
                _, loss, _ = model(inp, tgt)
                total_loss += loss.item(); n_done += 1
            if n_done >= n_batches:
                break
    finally:
        if ablate_engram:
            for blk in model.blocks:
                if blk.engram is not None:
                    eg = blk.engram
                    if id(eg) in original_forwards:
                        eg.forward = original_forwards[id(eg)]  # <2>

    return math.exp(total_loss / max(n_done, 1))


print("Evaluating (model is untrained — values are random-init baseline)")
ppl_with    = eval_perplexity(model_eg, ablate_engram=False)
ppl_without = eval_perplexity(model_eg, ablate_engram=True)
print(f"Perplexity WITH    Engram: {ppl_with:.2f}")
print(f"Perplexity WITHOUT Engram: {ppl_without:.2f}")
print(f"ΔPP: {ppl_without - ppl_with:+.2f} ({100*(ppl_without/ppl_with - 1):+.1f}%)")
print("\nNote: For a trained model, expect 2–8% perplexity increase when ablating.")

1. Monkey-patching `eg.forward` to return zeros cleanly ablates the Engram contribution without modifying model weights.
2. The `finally` block restores the original forward methods even if evaluation raises an exception, preventing permanent model corruption.

## Summary

| Component | Purpose | Key design |
|---|---|---|
| `CompressedTokenizer` | Normalize hash space | NFKC + lowercase + accent strip |
| `NgramHashMapping` | Map $n$-grams → table addresses | Seeded XOR + globally distinct prime moduli |
| `MultiHeadEmbedding` | Read hash addresses | Single table + per-head offsets |
| Gate | Route embedding into residual | sqrt-sigmoid of normalized dot-product |
| `ShortConv` | Local aggregation | kernel=4, dilation=3, depthwise, causal |
| `NanoEngram` | Full layer | hash → embed → gate → ShortConv → residual |
| mHC (simplified) | Key diversity | Collapsed to single $4d$ projection; full mHC requires Riemannian optimizer |

: {tbl-colwidths="[22,30,48]"}

## Exercises

1. **Collision rate measurement.** Estimate the theoretical collision rate for bigrams with table size $N = 50{,}000$ and GPT-2's vocabulary ($V = 50{,}257$). Then measure empirically: generate 10,000 random bigrams from the `NgramHashMapping` and count how many distinct bigrams share the same hash address.

2. **Normalization ablation.** Replace `_normalize` with a stricter version that also strips punctuation. Does this reduce the compressed vocabulary further? Plot the gate entropy before and after a short training run with each normalization to assess impact on routing signal quality.

3. **ShortConv ablation.** Remove the `ShortConv` from `NanoEngram` (return `gate * V` directly) and measure perplexity change on a trained checkpoint. Is local aggregation important, or does gating alone capture most of the benefit?

4. **sqrt-sigmoid vs sigmoid.** Plot both $\sigma(x)$ and $\sigma(\sqrt{|x|} \cdot \operatorname{sign}(x))$ over $x \in [-3, 3]$. How do they differ in the near-zero region? Implement a version of `NanoEngram` using plain sigmoid and compare gate entropy histograms after training.

5. **Multi-head diversity.** After training, compute the pairwise cosine similarity between the `n_head` hash head embeddings for the most common 1,000 bigrams. Are the heads learning diverse representations, or are they collapsing to similar embeddings?

:::{.callout-note}
## References
- DeepSeek-AI (2025). *DeepSeek-V3-Engram: Hash-Addressed n-gram Memory for Language Models.* Technical Report.
- DeepSeek-AI (2024). *DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model.* arXiv:2405.04434.
- Zhang & Sennrich (2019). *Root Mean Square Layer Normalization.* NeurIPS.

:::

■